In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [32]:

DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

calls = pd.read_excel(
    DATA_RAW / 'Calls (Done).xlsx',
    dtype={'Id': str, 'CONTACTID': str}
)

In [8]:
calls.shape

(95874, 11)

In [16]:
calls.describe()

,Call Duration (in seconds),Dialled Number,Scheduled in CRM,Tag
count,95791.000000,0.0,86875.000000,0.0
mean,164.977263,NaN,0.001635,NaN
std,401.410826,NaN,0.040397,NaN
min,0.000000,NaN,0.000000,NaN
25%,4.000000,NaN,0.000000,NaN
50%,8.000000,NaN,0.000000,NaN
75%,98.000000,NaN,0.000000,NaN
max,7625.000000,NaN,1.000000,NaN


In [13]:
calls.head()

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,NaN,Outbound,28.0,Attended Dialled,NaN,Completed,0.0,NaN
2,5805028000000764027,30.06.2023 08:59,John Doe,NaN,Outbound,24.0,Attended Dialled,NaN,Completed,0.0,NaN
3,5805028000000787003,30.06.2023 09:20,John Doe,5805028000000645014,Outbound,6.0,Attended Dialled,NaN,Completed,0.0,NaN
4,5805028000000768019,30.06.2023 09:30,John Doe,5805028000000645014,Outbound,11.0,Attended Dialled,NaN,Completed,0.0,NaN


In [10]:
calls.dtypes

Id                                str
Call Start Time                   str
Call Owner Name                   str
CONTACTID                         str
Call Type                         str
Call Duration (in seconds)    float64
Call Status                       str
Dialled Number                float64
Outgoing Call Status              str
Scheduled in CRM              float64
Tag                           float64
dtype: object

In [17]:
calls.columns.tolist()

['Id',
 'Call Start Time',
 'Call Owner Name',
 'CONTACTID',
 'Call Type',
 'Call Duration (in seconds)',
 'Call Status',
 'Dialled Number',
 'Outgoing Call Status',
 'Scheduled in CRM',
 'Tag']

In [14]:
calls.isna().sum()

Id                                0
Call Start Time                   0
Call Owner Name                   0
CONTACTID                      3933
Call Type                         0
Call Duration (in seconds)       83
Call Status                       0
Dialled Number                95874
Outgoing Call Status           8999
Scheduled in CRM               8999
Tag                           95874
dtype: int64

In [19]:
calls.nunique() 

Id                            95874
Call Start Time               68445
Call Owner Name                  33
CONTACTID                     15214
Call Type                         3
Call Duration (in seconds)     2619
Call Status                      11
Dialled Number                    0
Outgoing Call Status              4
Scheduled in CRM                  2
Tag                               0
dtype: int64

In [34]:
categorical_cols = ['Id',
 'Call Start Time',
 'Call Owner Name',
 'CONTACTID',
 'Call Type',
 'Call Duration (in seconds)',
 'Call Status',
 'Dialled Number',
 'Outgoing Call Status',
 'Scheduled in CRM',
 'Tag'
]

for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(calls[col].value_counts(dropna=False))


=== Id ===
Id
5805028000000805001    1
5805028000000768006    1
5805028000000764027    1
5805028000000787003    1
5805028000000768019    1
                      ..
5805028000056889515    1
5805028000056875317    1
5805028000056832495    1
5805028000056893619    1
5805028000056893631    1
Name: count, Length: 95874, dtype: int64

=== Call Start Time ===
Call Start Time
06.06.2024 15:07    9
09.04.2024 14:23    8
30.01.2024 12:12    7
18.03.2024 14:29    7
02.04.2024 18:15    7
                   ..
21.06.2024 15:22    1
21.06.2024 15:23    1
21.06.2024 15:26    1
21.06.2024 15:29    1
21.06.2024 15:31    1
Name: count, Length: 68445, dtype: int64

=== Call Owner Name ===
Call Owner Name
Yara Edwards       9059
Julia Nelson       7446
Ian Miller         7215
Charlie Davis      7213
Diana Evans        6857
Ulysses Adams      6085
Amy Green          5982
Nina Scott         5581
Victor Barnes      5439
Kevin Parker       5406
Paula Underwood    4580
Quincy Vincent     4384
Jane Smith      

Выводы по первичному ознакомлению:
По типам - мы преобразовали сразу Id и CONTACTID в строку, 
Call Start Time - в формат дата - время
Call Duration - оставляем в флоат
Scheduled in CRM -значения 1 и 0, это флаг, но оставляем как есть чтоб не заменились наны на тру

По данным - колонки Tag и Dialled Number пустые, можно удалить

по пропускам
CONTACTID - оставляем как есть
Tag и Dialled Number - удаляем колонки полностью
Outgoing Call Status  и Scheduled in CRM заполнить не можем

## Первичный осмотр Calls — выводы

### Размер
- 95 874 звонка × 11 колонок
- В 4.4 раза больше Deals — детальный лог звонков менеджеров

### Преобразование типов

| Колонка | Действие |
|---------|----------|
| `Id`, `CONTACTID` | str — уже сделано при загрузке |
| `Call Start Time` | → datetime (формат `DD.MM.YYYY HH:MM`) |
| `Call Duration (in seconds)` | оставить float (поддержка NaN) |
| `Scheduled in CRM` | → bool (значения 0/1) | - но оставим как есть 

### Колонки на удаление

- **`Tag`** — 100% пропусков (95874 NaN), артефакт выгрузки
- **`Dialled Number`** — 100% пропусков (95874 NaN), артефакт выгрузки

### Решения по пропускам

| Колонка | Пропусков | Решение |
|---------|-----------|---------|
| `CONTACTID` | 3933 (4%) | оставить NaN — звонки без привязки к контакту |
| `Call Duration` | 83 | оставить NaN |
| `Outgoing Call Status` | 8999 | оставить NaN — заполнено только для исходящих звонков (Outbound), для Inbound/Missed по определению пусто |
| `Scheduled in CRM` | 8999 | аналогично — только для исходящих |

**Подтверждение логики пропусков Outgoing/Scheduled:**  
Inbound (3078) + Missed (5921) = 8999 — точно совпадает с количеством пропусков. Это бизнес-логика, не баг.

### Наблюдения для отчёта

- **22% званков с Duration = 0** (21 975 шт) — "набрал-сбросил" или не дозвонился до гудка. Индикатор качества контактных данных.
- **33 уникальных Call Owner** против 27 в Deals — часть менеджеров делает звонки, но не ведёт сделки (вероятно, junior-уровень / call-center).
- **Распределение типов:** Outbound 91%, Missed 6%, Inbound 3% — школа в основном звонит сама, не принимает входящие.

In [35]:
calls['Call Start Time'] = pd.to_datetime(calls['Call Start Time'], format='%d.%m.%Y %H:%M')
calls = calls.drop(columns=['Tag', 'Dialled Number'])

In [36]:
print(calls['Scheduled in CRM'].value_counts(dropna=False))

Scheduled in CRM
0.0    86733
NaN     8999
1.0      142
Name: count, dtype: int64


In [37]:
calls.dtypes

Id                                       str
Call Start Time               datetime64[us]
Call Owner Name                          str
CONTACTID                                str
Call Type                                str
Call Duration (in seconds)           float64
Call Status                              str
Outgoing Call Status                     str
Scheduled in CRM                     float64
dtype: object

In [29]:
calls.shape

(95874, 11)

Сделаем флаг - Ответили или не ответили на звонок

In [39]:
# Маппинг статусов: ответили (True) / не ответили (False)
attended_mapping = {
    # Ответили
    'Attended Dialled': True,
    'Received': True,
    'Scheduled Attended': True,
    'Scheduled Attended Delay': True,
    # Не ответили
    'Unattended Dialled': False,
    'Missed': False,
    'Cancelled': False,
    'Overdue': False,
    'Scheduled Unattended': False,
    'Scheduled Unattended Delay': False,
    'Scheduled': False,
}

calls['is_attended'] = calls['Call Status'].map(attended_mapping)

# Проверка
print(calls['is_attended'].value_counts(dropna=False))
print(f"\nДоля отвеченных звонков: {calls['is_attended'].sum() / len(calls) * 100:.1f}%")

is_attended
True     73816
False    22058
Name: count, dtype: int64

Доля отвеченных звонков: 77.0%


In [40]:
calls['is_zero_duration'] = calls['Call Duration (in seconds)'] == 0
print(f"Звонков с Duration = 0: {calls['is_zero_duration'].sum()}")
print(f"Доля от всех звонков: {calls['is_zero_duration'].sum() / len(calls) * 100:.1f}%")

Звонков с Duration = 0: 21975
Доля от всех звонков: 22.9%


# Проверка на дубликаты

In [42]:
print(f"Дубликатов по Id: {calls['Id'].duplicated().sum()}")
print(f"Полных дубликатов строк: {calls.duplicated().sum()}")

Дубликатов по Id: 0
Полных дубликатов строк: 0


# Сохранение

In [43]:
calls.to_parquet(DATA_PROCESSED / 'calls_clean.parquet')
print(f"✅ Parquet сохранён")

calls.to_excel(DATA_PROCESSED / 'calls_clean.xlsx', index=False)
print(f"✅ Excel сохранён")

✅ Parquet сохранён
✅ Excel сохранён
